In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
from datetime import timedelta
#dodato za zadatak 14.

In [0]:
#1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
#get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop API'
#get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    return data



In [0]:
#2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next линкове и враћа записе са свих страна у једној листи
#Помоћ: urljoin(url, page['next'])
#Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url: str) -> list:

    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])   
        if page['next'] is not None:
            url = urljoin(url, page['next']) 
        else:
            url = None    

    return records
# 'trenutna stranica' = url; petlja staje kad 'next' postane None -> url postaje None -> while None: False


In [0]:
#3. Чување сирових података. Написати функцију save_json(records: list, path: str) -> None која листу речника уписује у JSON фајл без икаквих измена.
#json.dump(records, f, ensure_ascii=False, indent=2)
#save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл data/raw/orders.json


def save_json(records: list, path: str) -> None:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False, indent=2)



# ensure_ascii=False - dozvoljava srpska slova (c,s,z) da ostanu prava slova u fajlu, ne \u kodovi


In [0]:
#4. Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити у задатку 5

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(path, 'wb') as f: 
        f.write(response.content) 

    return len(response.content)




In [0]:
#5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams', 'matches'

#5.1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json) - rešeno
#5.2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције download_file 
#5.3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
#Након тога преименовати колоне
#customer_name у customer
#customer_city у city

def extract(raw_dir: str) -> dict:

    #preuzima sva 4 izvora u raw_dir

    teams = download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir,'teams.csv')) #5.2
    products = download_file(FILES_URL + 'products.csv', os.path.join(raw_dir,'products.csv'))
    matches = download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir,'matches.parquet'))

    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json')) #5.1

    #5.3 vraća 4 df:
    # pd.json_normalize "spljoska" ugnjezdene recnike (customer.name -> customer_name) u ravne kolone

    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename(columns={'customer_name': 'customer', 'customer_city': 'city'})

    #5.4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
    #По потреби, pd.read_csv(...) или pd.read_parquet(...)

    teams_df = pd.read_csv(os.path.join(raw_dir,'teams.csv'))
    products_df = pd.read_csv(os.path.join(raw_dir,'products.csv'))
    matches_df = pd.read_parquet(os.path.join(raw_dir,'matches.parquet'))
    

    return  {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df,
    }
    


#5.5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)
data = extract(RAW_DIR)
print(data['orders'].shape)
print(data['products'].shape)
print(data['teams'].shape)
print(data['matches'].shape)

(1852, 8)
(108, 7)
(20, 5)
(120, 9)


In [0]:
#6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame која чисти каталог производа:

#6.1 Ако се понавља исти производ, садржати први ред за сваки product_id
#6.2 team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред остаје празан (то су производи лиге, без тима)
#6.3 category треба да буде без размака, има мала слова
#6.4 price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане само float (у сваком случају) Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
#6.5 active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало постаје False

#Output: 105 редова, price је типа float64, active је True за 101 производ
#Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves', 'price': 1490.0, 'active': True, ...


def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    is_dup = df.duplicated('product_id', keep='first') #6.1 remove duplicates

    df['team_code'] = df['team_code'].str.strip().str.upper() #6.2 team_code čišćenje
    df['category'] = df['category'].str.strip().str.lower() #6.3 category čišćenje
    df['price'] = df['price'].str.replace(' RSD', '', regex=False).str.replace(',', '.', regex=False) #6.4 price čišćenje
    df['price'] = pd.to_numeric(df['price']) #samostalna funkcija iz pandas modula

    df['active'] = df['active'].str.strip().str.lower() == 'yes' #6.5 aktive čišćenje i poredjenje svakog elementa sa == 'yes' da bi dobili True/False
    
    return df[~is_dup]



In [0]:
#7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame

#7.1. order_id, customer, city, payment без размака;
#7.2 order_id велика слова
#7.3 customer и city -> title()
#7.4 Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()  # da ne menjamo original van funkcije
    for col in ['order_id', 'customer', 'city', 'payment']:
        df[col] = df[col].str.strip()  #7.1

    df['order_id'] = df['order_id'].str.upper() #7.2
    df['customer'] = df['customer'].str.title() #7.3
    df['city'] = df['city'].str.title().fillna('Unknown') #7.3 i #7.4

    return df



In [0]:
#8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple која враћа (unique, duplicates) по order_id, задржава први
#Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return df[~is_dup], df[is_dup]
# keep='first' - prvo pojavljivanje NIJE duplikat; ~ je NOT (negacija) boolean maske




In [0]:
# 9. validate_order - ista logika kao mini projekat, samo pandas-ovi tipovi
#Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа 'OK' за исправан ред, у супротном разлог:

#Провере радити редом и вратити први пронађени проблем:
#order_id не почиње са NSS- -> 'invalid_order'
#quantity недостаје -> 'missing_quantity'
#quantity није цео број већи од нуле -> 'invalid_quantity'
#unit_price није број већи од нуле -> 'invalid_price'
#product_id не постоји међу производима -> 'unknown_product'

#Помоћ:
#pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
#pd.isna(value) препознаје и None и NaN

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')  # 'abc' -> NaN umesto greske
    price = pd.to_numeric(row['unit_price'], errors='coerce')
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(price) or price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    return 'OK'



In [0]:
#10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame
#Типови:
#quantity -> int
#unit_price -> float
#order_date -> датум (pd.to_datetime)
#Нова колона total (quantity * unit_price)
#Из очишћених производа (products) додати product_name, team_code и category (merge по product_id)
#Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price']).astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']
    names = products[['product_id', 'product_name', 'team_code', 'category']]
    return df.merge(names, on='product_id', how='left')
# merge = SQL JOIN analogija - svaka porudzbina "nasledi" ime/kategoriju proizvoda preko product_id

In [0]:
#11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,rejected):

#valid су трансформисани исправни редови
#rejected су очишћени неисправни редови и дупликати, са колоном reason (дупликати за разлог треба да имају 'duplicate')
#Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = clean_orders(df)
    unique, duplicates = remove_duplicate_orders(df)
    duplicates = duplicates.assign(reason='duplicate')
    product_ids = set(products['product_id'])
    reasons = unique.apply(lambda row: validate_order(row, product_ids), axis=1)
    valid = transform_orders(unique[reasons == 'OK'], products)
    rejected = pd.concat([unique[reasons != 'OK'].assign(reason=reasons[reasons != 'OK']), duplicates])
    return valid, rejected



In [0]:
#12. Резултати тимова. Написати функцију team_results(matches: pd.DataFrame, teams:pd.DataFrame) -> pd.DataFrame која сваку одиграну утакмицу претвара у два реда, једанза домаћина и један за госта.

#Користити само утакмице са status == 'finished' (одложена нема резултат)
#Колоне: 
# match_id, 
# match_date: (Датум из kickoff_ts , без времена - df['kickoff_ts'].dt.normalize())
# team_code,
#opponent_code
#goals_for,
#goals_against
#result('W','D','L'),
# points(3, 1, 0)

#Утакмице имају team_id, а производи team_code
#HINT: codes =dict(zip(teams['team_id'], teams['code']))   
# Прави речник облика {1: 'NPM', 2: 'FWR', ...}

def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    played = matches.copy()
    played = played.loc[(played['status'] == 'finished')]
    played['match_date'] = played['kickoff_ts'].dt.normalize()
    codes = dict(zip(teams['team_id'], teams['code']))

    home = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['match_date'],
        'team_code': played['home_team_id'].map(codes),
        'opponent_code': played['away_team_id'].map(codes),
        'goals_for': played['home_goals'],
        'goals_against': played['away_goals'],
    })

    away = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['match_date'],
        'team_code': played['away_team_id'].map(codes),
        'opponent_code': played['home_team_id'].map(codes),
        'goals_for': played['away_goals'],
        'goals_against': played['home_goals'],
    })

    results = pd.concat([home, away], ignore_index=True)

    results['result'] = 'D' #default vrednost za SVE redove (nerešeno), jer je to slučaj kad nijedan od druga dva uslova nije ispunjen.
    results.loc[results['goals_for'] > results['goals_against'], 'result'] = 'W'
    results.loc[results['goals_for'] < results['goals_against'], 'result'] = 'L'

    results['points'] = 0
    results.loc[results['result'] == 'W', 'points'] = 3
    results.loc[results['result'] == 'D', 'points'] = 1

    return results



In [0]:
#13. Табела лиге. Написати функцију league_table(results: pd.DataFrame) -> pd.DataFrame која од резултата из задатка 12 прави табелу
#Колоне:
#position,
#team_code,
#played,
#won,
#drawn,
#lost,
#goals_for,
#goals_against,
#goal_diff,
#points

#Редослед табеле: Бодови, па гол разлика, па дати голови (све опадајуће)
#Output: 20 тимова, први FWR (26 бодова), други BMC(23)
#HGA и OKT имају по 11 утакмица

def league_table(results: pd.DataFrame) -> pd.DataFrame:
    played = results.groupby('team_code').size()
    won = results.loc[results['result'] == 'W'].groupby('team_code').size()
    drawn = results.loc[results['result'] == 'D'].groupby('team_code').size()
    lost = results.loc[results['result'] == 'L'].groupby('team_code').size()
    goals_for = results.groupby('team_code')['goals_for'].sum()
    goals_against = results.groupby('team_code')['goals_against'].sum()

    table = pd.DataFrame({
    'team_code': played.index,
    'played': played,
    'won': won,
    'drawn': drawn,
    'lost': lost,
    'goals_for': goals_for,
    'goals_against': goals_against
})
    table['goal_diff'] = goals_for - goals_against
    table['points'] = won * 3 + drawn * 1

    table = table.sort_values(['points', 'goal_diff', 'goals_for'], ascending = False)

    table = table.reset_index(drop=True)
    table['position'] = table.index + 1

    return table




In [0]:
#14. Прозори после утакмица. Написати функцију
#match_windows(results: pd.DataFrame) ->pd.DataFrame
#која за сваки ред из задатка 12 прави 4 реда: дан утакмице и три дана послење, са резултатом те утакмице



def match_windows(results: pd.DataFrame) -> pd.DataFrame:
    sve_kopije = []

    for d in range(4):
        kopija = pd.DataFrame({
            'team_code': results['team_code'],
            'date': results['match_date'] + timedelta(days=d),
            'result': results['result']
        })
        sve_kopije.append(kopija)

    windows = pd.concat(sve_kopije, ignore_index=True)
    windows = windows.rename(columns={'result': 'window'})

    return windows



In [0]:
#15. Промет по колони. Написати функцију revenue_by(df: pd.DataFrame, column: str) ->pd.DataFrame
#Kолоне [column, 'revenue'] , сортирано опадајуће
#Input: valid, 'team_code' -> Output: 20 редова, први LSB 720420.00, последњи WFT 234650.00
#Input: valid, 'category' → Output: jerseys 7094410.00 на првом месту

def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:
    result = df.groupby(column)['total'].sum().round(2).reset_index(name='revenue')
    return result.sort_values('revenue', ascending=False).reset_index(drop=True)

In [0]:
#16. Продаја после резултата. Написати функцију sales_after_results(valid: pd.DataFrame,windows: pd.DataFrame) -> pd.DataFrame
#која одговара на питање пројекта:
#16.1. Календар: Сваки тим x Сваки дан.
#16.2. Дневни промет по тиму и дану
#16.3. Спојити календар са прозорима добијених из match_windows
#16.4. Дан који није ни у једном прозору нема резултат и добија 'none'
#16.5. Груписати по window:team_days(број дана),revenue(збир),avg_daily_revenue(просек, 2 децимале), сортирано опадајуће

def sales_after_results(valid: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    # 16.1. kalendar: svaki tim x svaki dan
    teams_list = pd.DataFrame({'team_code': windows['team_code'].unique()})
    days_list = pd.DataFrame({'order_date': valid['order_date'].unique()})
    calendar = teams_list.merge(days_list, how='cross')

    # 16.2. dnevni promet po timu i danu
    dnevni_promet = valid.groupby(['team_code', 'order_date'])['total'].sum().reset_index(name='revenue')
    calendar = calendar.merge(dnevni_promet, on=['team_code', 'order_date'], how='left')
    calendar['revenue'] = calendar['revenue'].fillna(0)

    # 16.3. spajanje sa prozorima
    calendar = calendar.merge(
        windows,
        left_on=['team_code', 'order_date'],
        right_on=['team_code', 'date'],
        how='left'
    )

    # 16.4. dan bez prozora dobija 'none'
    calendar['window'] = calendar['window'].fillna('none')

    # 16.5. finalno grupisanje po window
    team_days = calendar.groupby('window').size()
    revenue = calendar.groupby('window')['revenue'].sum().round(2)
    avg_daily_revenue = calendar.groupby('window')['revenue'].mean().round(2)

    rezultat = pd.DataFrame({
        'window': team_days.index,
        'team_days': team_days,
        'revenue': revenue,
        'avg_daily_revenue': avg_daily_revenue
    })

    rezultat = rezultat.sort_values('revenue', ascending=False)
    rezultat = rezultat.reset_index(drop=True)

    return rezultat



In [0]:
#17. Load. Написати функцију load(outputs: dict, processed_dir: str) -> dict која сваки DataFrame из речника {име_фајла: DataFrame} уписује у processed_dir
#и враћа {име_фајла: број_редова}

def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
        written[name] = len(df)
    return written


In [0]:
#18. Написати функцију run_pipeline(raw_dir: str, processed_dir: str) -> dict која покреће цео pipeline

def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric', 'value']).to_csv(
        os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary

In [0]:
summary = run_pipeline(RAW_DIR, PROCESSED_DIR)
summary

{'total_rows': 1852,
 'duplicates': 12,
 'valid_rows': 1800,
 'rejected_rows': 52,
 'total_revenue': 10045400.0,
 'files_written': 8}